# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Preview metadata
print("Dataset loaded!")
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available Record Sets, Fields, and their IDs. All are referenced by `@id`.

In [ ]:
# List available record sets and their field @ids
print("Record Sets:")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"\nRecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name}, @id: {field.id}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s discovered in the previous step.

In [ ]:
# Extract all available record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Use only `@id` for all fields.

In [ ]:
# As an example, select the first record set and look for numeric fields
import numpy as np
if len(dataframes) > 0:
    primary_record_set_id = list(dataframes.keys())[0]
    df = dataframes[primary_record_set_id]

    # Let's attempt to find a numeric field from the Field metadata
    rs = next(rs for rs in dataset.metadata.record_sets if rs.id == primary_record_set_id)
    numeric_field_id = None
    for field in rs.fields:
        # Typical numeric data types: Float, Integer, Number
        if field.data_type and field.data_type.lower() in ["float", "integer", "number"]:
            if field.id in df.columns:
                numeric_field_id = field.id
                break

    if numeric_field_id:
        # Show distribution summary
        print(f"Numeric field for analysis: {numeric_field_id}")
        print(df[numeric_field_id].describe())
        
        # Remove nulls then filter
        df_num = df[df[numeric_field_id].notnull()]
        df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id], errors="coerce")

        threshold = np.nanpercentile(df_num[numeric_field_id], 75)  # top 25%
        filtered_df = df_num[df_num[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a grouping field (categorical)
        group_field_id = None
        for field in rs.fields:
            # Choose a non-numeric, non-ID field
            if field.data_type and field.data_type.lower() not in ["float", "integer", "number"] and field.id in df.columns and field.id != numeric_field_id:
                group_field_id = field.id
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}, showing mean {numeric_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No dataframes available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using available numeric and, if possible, categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'df_num' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df_num[numeric_field_id].dropna(), bins=20, kde=True, color='teal')
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_num)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric fields available for visualization.')

## 6. Conclusion
This notebook has guided you through:
- Accessing and loading Croissant FAIR^2 dataset metadata
- Discovering and extracting data using record set and field `@id`
- Exploring data structure and performing EDA on selected numeric fields
- Visualizing field distributions

For a full, detailed analysis, consult the field and record set `@id`s above and the [mlcroissant documentation](https://mlcommons.org/croissant/).

_Note: Not all datasets may include record sets with records or numeric fields suitable for EDA; adjust selection logic as needed for your dataset._